## 내 Codex 하네스 만들기

이전 판의 여섯 가지 선택 실습을 같은 로컬 Git 저장소에서 이어 간다. WSL과 macOS에서 전체 묶음을 풀고 daily_resources 옆에서 노트북을 연다. 설치한 Codex CLI로 로그인하고 git, Python, mcp 패키지를 준비한다. 각 단계는 노트북을 실행한 다음 로컬 Codex에서 직접 시험해야 끝난다.

## 이전 판 실습 080 · 내 Codex 저장소와 재현 조건 만들기

예상 120분. 자료를 준비하고 입력을 예측하는 데 15분, 첫 구현에 50분, 다른 입력과 실패 사례에 35분, 결과를 다시 확인하는 데 20분을 씁니다.

**준비물** 로컬 Codex CLI, Git, Jupyter, daily_resources/codex-kit. 80~85일은 같은 my-codex-desk를 사용한다.

[전체 실습 묶음](https://langchain-lecture.vercel.app/daily/all-labs.zip)을 풀고 이 노트북을 daily_resources 옆에 두세요. [오늘의 수업 예제](https://langchain-lecture.vercel.app/chapter80/index.html)

### 내 Codex 작업 저장소와 실패 기준 만들기

WSL 또는 macOS에서 노트북과 daily_resources를 같은 폴더에 둔다. 이 셀은 그 옆에 수업용 Git 저장소를 만든다. 빈 저장소에서 시작하려면 다른 폴더에 실습 묶음을 푼다. 노트북의 Python과 Codex를 같은 운영체제 안에서 사용한다.

In [ ]:
from pathlib import Path
import sys, json, shutil, subprocess
kit = Path('daily_resources/codex-kit').resolve()
repo = Path('my-codex-desk').resolve()
repo.mkdir(exist_ok=True)
def create_file(relative, content):
    target = repo / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_text(content, encoding='utf-8')
    return target
subprocess.run(['git','init','-q'],cwd=repo,check=True)
for name in ('tasks.json','todo.py','test_todo.py'):
    create_file(name,(kit/name).read_text(encoding='utf-8'))
create_file('AGENTS.md','동명 항목을 ID로 구분한다. 변경 전에 재현하고 변경 뒤 같은 테스트를 실행한다.\n')
baseline = subprocess.run([sys.executable,'-m','unittest','-v'],cwd=repo,capture_output=True,text=True)
print(repo,baseline.returncode,baseline.stderr[-550:])

시작 테스트는 ID 2 대신 ID 1이 완료되어 실패한다. 터미널에서 출력된 경로로 이동해 codex를 열고 “파일을 고치기 전에 두 ID의 기대·실제 상태를 설명해 줘”라고 요청한다. 학생이 직접 세 번째 동명 항목 조건을 AGENTS.md에 추가한다.

### 납품할 결과

동명 업무를 잘못 완료하는 결함을 다음 날에도 그대로 재현할 Git 저장소를 만듭니다.

**구현 단서** 교재 셀로 시작 파일을 만든 후 tests에서 done 값을 모두 false로 초기화한 이유를 확인하세요. 실제 완료 ID와 기대 ID를 README에 기록하고 기준 상태를 커밋합니다.

### 1. 구현하기

로컬 Git 저장소에 동명 할 일 두 건과 실패하는 테스트를 넣고 AGENTS.md에 변경 범위를 쓴다.

In [ ]:
tasks = json.loads((repo/'tasks.json').read_text(encoding='utf-8'))
print([(t['id'],t['title'],t['done']) for t in tasks])
print('기준 테스트 종료 코드', baseline.returncode)
# 수정 전 기준 상태를 커밋한 뒤 새 세션에 ID 2의 기대 상태와 실제 상태를 설명해 달라고 요청하세요.

### 2. 조건을 바꿔 다시 실행하기

Codex에 수정 전 상태를 읽게 하고 세 번째 동명 항목을 추가해 보존할 ID를 정한다.

In [ ]:
# id=2 요청 시 실제 [1], 기대 [2]가 보여야 합니다. 이후 스킬이 수정 전 상태를 읽을 수 있게 이 커밋 번호를 남기세요.
# 실험 조건, 예상값, 실제 반환값과 근거 파일을 아래에 기록하세요.

### 3. 스스로 확인하기

실패한 테스트의 실제 ID와 기대 ID를 적고 학생이 작성한 지침을 저장한다.

**실패 원인 좁히기** id=2 요청 시 실제 [1], 기대 [2]가 보여야 합니다. 이후 스킬이 수정 전 상태를 읽을 수 있게 이 커밋 번호를 남기세요.

**내 기록**

- 처음 사용한 입력과 기대값
- 실제 실행 결과와 달랐던 지점
- 수정한 줄 또는 설정, 다시 실행한 결과

## 이전 판 실습 081 · 저장소 전용 Codex 스킬 작성

예상 120분. 자료를 준비하고 입력을 예측하는 데 15분, 첫 구현에 50분, 다른 입력과 실패 사례에 35분, 결과를 다시 확인하는 데 20분을 씁니다.

**준비물** 로컬 Codex CLI, Git, Jupyter, daily_resources/codex-kit. 80~85일은 같은 my-codex-desk를 사용한다.

[전체 실습 묶음](https://langchain-lecture.vercel.app/daily/all-labs.zip)을 풀고 이 노트북을 daily_resources 옆에 두세요. [오늘의 수업 예제](https://langchain-lecture.vercel.app/codex81/index.html)

### Codex가 발견할 프로젝트 스킬 작성하기

80일 저장소를 연다. 언제 이 작업 절차를 읽어야 하는지는 description에 쓰고, 테스트 순서는 본문에 쓴다. 아래 초안을 만든 뒤 누락된 ID의 기대 동작과 학생 자신의 회귀 기준을 본문에 직접 추가한다.

In [ ]:
from pathlib import Path
import sys, json, shutil, subprocess
kit = Path('daily_resources/codex-kit').resolve()
repo = Path('my-codex-desk').resolve()
repo.mkdir(exist_ok=True)
def create_file(relative, content):
    target = repo / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_text(content, encoding='utf-8')
    return target
skill = create_file('.agents/skills/todo-change/SKILL.md','---\nname: todo-change\ndescription: 할 일 완료나 삭제가 잘못된 ID에 적용되는 결함을 재현하고 수정할 때 사용한다.\n---\n1. 동명 항목 두 개를 준비하고 선택한 ID와 유지할 ID를 적는다.\n2. 작업 폴더에서 python -m unittest -v 로 실패한 검사를 확인한다.\n3. 변경 후 같은 검사와 기존 행을 확인한다. 실행 결과와 바뀐 파일을 남긴다.\n')
print(skill.read_text())

저장소 루트에서 새 Codex 세션을 열어 “$todo-change 동명 항목 완료 결함을 재현만 해줘”라고 요청한다. 테스트 이름과 실패한 ID를 기록한다. 이어 평범한 문서 제목 변경 요청을 보내 스킬을 언제 명시해야 하는지 비교한다.

[공식 문서](https://developers.openai.com/codex/skills)

### 납품할 결과

todo-change를 처음 보는 사람도 재현부터 확인까지 수행할 저장소 전용 스킬을 만듭니다.

**구현 단서** description에는 언제 선택할지를, 본문에는 준비할 입력·명령·작업 중지 조건을 적으세요. 없는 ID 999의 동작을 정한 뒤 명시적으로 스킬을 호출하고 문서 읽기만 하는 요청과 비교합니다.

### 1. 구현하기

todo-change 스킬의 이름·발동 조건·테스트 절차를 정의하고 본문에 누락 ID 규칙을 추가한다.

In [ ]:
skill_text = (repo/'.agents/skills/todo-change/SKILL.md').read_text(encoding='utf-8')
print(skill_text.split('---')[1].strip())
# ID 999의 처리 원칙을 본문에 추가한 뒤, 새 세션에서 $todo-change로 두 ID를 시험하세요.

### 2. 조건을 바꿔 다시 실행하기

새 Codex 세션에서 스킬을 명시한 수정 요청과 단순 문서 변경을 비교한다.

In [ ]:
# 스킬이 발동했다는 대답 외에 어느 테스트를 실제로 실행했고 어느 ID가 실패했는지 세션에서 찾아 적으세요.
# 실험 조건, 예상값, 실제 반환값과 근거 파일을 아래에 기록하세요.

### 3. 스스로 확인하기

실제로 선택한 스킬 이름과 테스트 출력이 남는다.

**실패 원인 좁히기** 스킬이 발동했다는 대답 외에 어느 테스트를 실제로 실행했고 어느 ID가 실패했는지 세션에서 찾아 적으세요.

**내 기록**

- 처음 사용한 입력과 기대값
- 실제 실행 결과와 달랐던 지점
- 수정한 줄 또는 설정, 다시 실행한 결과

## 이전 판 실습 082 · Codex 세션 훅을 직접 연결

예상 120분. 자료를 준비하고 입력을 예측하는 데 15분, 첫 구현에 50분, 다른 입력과 실패 사례에 35분, 결과를 다시 확인하는 데 20분을 씁니다.

**준비물** 로컬 Codex CLI, Git, Jupyter, daily_resources/codex-kit. 80~85일은 같은 my-codex-desk를 사용한다.

[전체 실습 묶음](https://langchain-lecture.vercel.app/daily/all-labs.zip)을 풀고 이 노트북을 daily_resources 옆에 두세요. [오늘의 수업 예제](https://langchain-lecture.vercel.app/codex82/index.html)

### 세션 시작 훅을 설치하고 실제 이벤트 구별하기

훅은 이벤트가 발생할 때 프로그램이 호출한다. 함수만 수동 실행해서 얻은 로그와 새 Codex 세션이 남긴 로그를 따로 관찰한다. 명령이 가리키는 스크립트를 읽고 이 저장소에서 사용할 훅 정의를 검토한다.

In [ ]:
from pathlib import Path
import sys, json, shutil, subprocess
kit = Path('daily_resources/codex-kit').resolve()
repo = Path('my-codex-desk').resolve()
repo.mkdir(exist_ok=True)
def create_file(relative, content):
    target = repo / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_text(content, encoding='utf-8')
    return target
create_file('session_event.py',(kit/'session_event.py').read_text())
config = {'hooks': {'SessionStart': [{'hooks': [{'type': 'command', 'command': 'python3 "$(git rev-parse --show-toplevel)/session_event.py"'}]}]}}
create_file('.codex/hooks.json',json.dumps(config,indent=2))
subprocess.run([sys.executable,'session_event.py'],cwd=repo,input=json.dumps({'hook_event_name':'SessionStart','source':'fixture'}),text=True,check=True)
print((repo/'hook-events.jsonl').read_text().splitlines()[-1])

fixture 한 줄은 수동 시험이다. 프로젝트 루트에서 로컬 codex를 열고 /hooks에서 명령과 경로를 확인해 해당 정의를 신뢰한다. 종료 후 새 세션을 열어 fixture와 다른 source를 가진 새 행을 찾는다. 내 훅에서 기록할 필드를 하나 바꾸고 새 세션으로 다시 확인한다.

[공식 문서](https://developers.openai.com/codex/hooks)

### 납품할 결과

새 Codex 세션이 열릴 때 발생하는 사건을 한 줄씩 남기는 훅을 작성합니다.

**구현 단서** 먼저 임시 입력 source=fixture로 스크립트를 실행해 로그 줄 수를 적습니다. 프로젝트 훅을 연결하고 새 세션을 열어 추가된 줄의 이벤트 이름과 source를 비교하세요.

### 1. 구현하기

SessionStart 이벤트를 받는 스크립트와 프로젝트 hooks.json을 만들고 수동 이벤트로 점검한다.

In [ ]:
log = repo/'hook-events.jsonl'
with log.open(encoding='utf-8') as f:
    recent = [json.loads(line) for line in f if line.strip()]
print('현재 로그 행 수',len(recent),recent[-1])
# 이 수를 먼저 기록한 뒤 새 Codex 세션을 열고 셀을 다시 실행하세요.

### 2. 조건을 바꿔 다시 실행하기

정의를 검토한 후 새 로컬 세션을 열어 로그 한 행을 확인하고 기록할 필드를 한 개 바꾼다.

In [ ]:
# 수동 실행과 새 세션의 두 행을 따로 보관하세요. 로그가 늘지 않으면 프로젝트 신뢰 상태, 훅 등록 경로, 명령의 작업 폴더를 차례로 확인합니다.
# 실험 조건, 예상값, 실제 반환값과 근거 파일을 아래에 기록하세요.

### 3. 스스로 확인하기

fixture와 실제 세션의 source 및 새 로그 행을 구분한다.

**실패 원인 좁히기** 수동 실행과 새 세션의 두 행을 따로 보관하세요. 로그가 늘지 않으면 프로젝트 신뢰 상태, 훅 등록 경로, 명령의 작업 폴더를 차례로 확인합니다.

**내 기록**

- 처음 사용한 입력과 기대값
- 실제 실행 결과와 달랐던 지점
- 수정한 줄 또는 설정, 다시 실행한 결과

## 이전 판 실습 083 · Codex 서브에이전트로 재현 결과 인계

예상 120분. 자료를 준비하고 입력을 예측하는 데 15분, 첫 구현에 50분, 다른 입력과 실패 사례에 35분, 결과를 다시 확인하는 데 20분을 씁니다.

**준비물** 로컬 Codex CLI, Git, Jupyter, daily_resources/codex-kit. 80~85일은 같은 my-codex-desk를 사용한다.

[전체 실습 묶음](https://langchain-lecture.vercel.app/daily/all-labs.zip)을 풀고 이 노트북을 daily_resources 옆에 두세요. [오늘의 수업 예제](https://langchain-lecture.vercel.app/codex83/index.html)

### 읽기 전용 서브에이전트에 오류 재현을 위임하기

부모는 수정 권한과 통합 테스트를 맡는다. 자식은 파일을 읽어 입력·기대 상태·근거 위치만 돌려준다. 세션을 열기 전에 학생이 인계 지침에 “실제 상태”와 그 증거를 추가한다.

In [ ]:
from pathlib import Path
import sys, json, shutil, subprocess
kit = Path('daily_resources/codex-kit').resolve()
repo = Path('my-codex-desk').resolve()
repo.mkdir(exist_ok=True)
def create_file(relative, content):
    target = repo / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_text(content, encoding='utf-8')
    return target
role = create_file('.codex/agents/case_reader.toml','name = "case_reader"\ndescription = "동명 항목의 상태 변화와 실패 테스트를 읽어 인계한다."\nsandbox_mode = "read-only"\ndeveloper_instructions = """\n파일 편집 없이 tasks.json과 test_todo.py를 읽는다.\n선택한 ID, 보존할 ID, 기대 상태 및 근거 파일을 반환한다.\n테스트를 실행하지 않았다면 그렇게 적는다.\n"""\n')
print(role.read_text())

새 Codex 세션에서 “case_reader 서브에이전트를 실제로 호출해 ID 2의 기대값과 근거 파일을 받아. 인계를 받을 때까지 수정하지 마”라고 요청한다. 자식 실행 기록과 반환값을 확인한다. 부모가 직접 같은 설명을 쓴 경우와 구별한다. 인계 후 부모에게 todo.py 한 파일만 고치게 하고 80일 테스트를 다시 돌린다.

[공식 문서](https://developers.openai.com/codex/subagents)

### 납품할 결과

읽기 전용 case_reader가 재현 내용을 부모에게 전달하고 부모만 코드를 수정하는 작업 기록을 만듭니다.

**구현 단서** 자식에게 기대 ID, 관찰 ID, 근거 파일, 테스트를 직접 실행했는지 네 필드를 요구하세요. 부모는 인계를 받은 뒤 todo.py의 선택 기준만 수정하고 같은 테스트를 실행합니다.

### 1. 구현하기

읽기 전용 case_reader를 작성하고 부모가 자식에게 동명 항목 재현을 위임하도록 요청한다.

In [ ]:
print((repo/'.codex/agents/case_reader.toml').read_text(encoding='utf-8'))
handoff_fields = ['선택 ID','기대 완료 ID','관찰 완료 ID','근거 파일','실행한 명령']
# 자식이 실제 돌려준 값으로 인계표를 작성한 뒤 부모의 변경 diff를 보세요.

### 2. 조건을 바꿔 다시 실행하기

인계 필드에 실제 상태와 근거를 추가하고 부모가 최소 수정 후 다시 테스트한다.

In [ ]:
# 자식의 실제 호출 기록과 반환 필드, 부모 diff, 통과한 테스트를 짝지으세요. 80일의 테스트가 이미 통과하면 기준 커밋의 별도 작업 폴더에서 결함을 재현하세요.
# 실험 조건, 예상값, 실제 반환값과 근거 파일을 아래에 기록하세요.

### 3. 스스로 확인하기

자식 호출 기록·반환한 두 ID·부모의 변경 파일과 통과 결과를 확인한다.

**실패 원인 좁히기** 자식의 실제 호출 기록과 반환 필드, 부모 diff, 통과한 테스트를 짝지으세요. 80일의 테스트가 이미 통과하면 기준 커밋의 별도 작업 폴더에서 결함을 재현하세요.

**내 기록**

- 처음 사용한 입력과 기대값
- 실제 실행 결과와 달랐던 지점
- 수정한 줄 또는 설정, 다시 실행한 결과

## 이전 판 실습 084 · Codex용 로컬 MCP 서버 등록

예상 120분. 자료를 준비하고 입력을 예측하는 데 15분, 첫 구현에 50분, 다른 입력과 실패 사례에 35분, 결과를 다시 확인하는 데 20분을 씁니다.

**준비물** 로컬 Codex CLI, Git, Jupyter, daily_resources/codex-kit. 80~85일은 같은 my-codex-desk를 사용한다.

[전체 실습 묶음](https://langchain-lecture.vercel.app/daily/all-labs.zip)을 풀고 이 노트북을 daily_resources 옆에 두세요. [오늘의 수업 예제](https://langchain-lecture.vercel.app/codex84/index.html)

### 자기 MCP 서버를 Codex 도구 목록에 등록하기

노트북 커널에 mcp 패키지가 있어야 한다. 제공된 get_task를 읽고 반환값에 요청한 ID를 직접 추가한다. 수업용 config.toml이 이미 있다면 아래 출력의 서버 블록만 기존 파일에 합친다. TOML 문자열 경로를 절대 경로로 생성하므로 공백이 있는 폴더에서도 동작한다.

In [ ]:
from pathlib import Path
import sys, json, shutil, subprocess
kit = Path('daily_resources/codex-kit').resolve()
repo = Path('my-codex-desk').resolve()
repo.mkdir(exist_ok=True)
def create_file(relative, content):
    target = repo / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_text(content, encoding='utf-8')
    return target
for name in ('catalog_server.py','probe.py'):
    create_file(name,(kit/name).read_text())
block = '[mcp_servers.todo_catalog]\ncommand = '+json.dumps(sys.executable,ensure_ascii=False)+'\nargs = ['+json.dumps(str(repo/'catalog_server.py'),ensure_ascii=False)+']\n'
create_file('.codex/config.toml',block)
print(block)
print(subprocess.run([sys.executable,'probe.py'],cwd=repo,capture_output=True,text=True,check=True).stdout)

프로토콜 검사에서 get_task, 2 found, 999 missing이 보인다. 저장소를 신뢰한 로컬 Codex의 새 세션에서 /mcp로 todo_catalog를 확인한다. “등록한 MCP 도구로 2번과 999번을 각각 조회해”라고 요청하고 실제 호출 기록을 연다. 파일을 직접 읽어 답한 경우에는 MCP 연결 검사를 통과한 것으로 세지 않는다.

[공식 문서](https://developers.openai.com/codex/mcp)

### 납품할 결과

내 작업 저장소만 읽는 get_task MCP 서버를 Codex에 연결하고 요청 ID가 포함된 조회 결과를 만듭니다.

**구현 단서** 정상 ID 2와 없는 ID 999를 먼저 probe.py로 호출한 뒤 실제 Codex 세션에서 다시 호출합니다. 서버 경로를 잘못 둔 시험에서는 도구 목록 발견부터 실패하는지 기록한 뒤 복구하세요.

### 1. 구현하기

읽기 전용 get_task 도구를 학생 저장소에서 실행하고 반환값에 요청 ID를 추가한다. 프로젝트 설정에 등록한다.

In [ ]:
print((repo/'.codex/config.toml').read_text(encoding='utf-8'))
for test_id in (2,999):
    print('Codex에서 get_task를 사용할 item_id',test_id)
# 두 도구 응답의 원본 JSON을 보관하고 각 응답에 요청 ID를 넣도록 서버를 수정하세요.

### 2. 조건을 바꿔 다시 실행하기

stdio 클라이언트와 실제 Codex에서 각각 ID 2와 999를 조회한다. 서버 경로 오류를 고친 뒤 재연결한다.

In [ ]:
# 네 응답에서 요청 ID와 found/missing을 대조하세요. 파일을 직접 읽어서 맞힌 답은 도구 호출 기록이 생기지 않습니다.
# 실험 조건, 예상값, 실제 반환값과 근거 파일을 아래에 기록하세요.

### 3. 스스로 확인하기

도구 목록에 get_task가 보이고 실제 호출에서 found와 missing이 구별된다.

**실패 원인 좁히기** 네 응답에서 요청 ID와 found/missing을 대조하세요. 파일을 직접 읽어서 맞힌 답은 도구 호출 기록이 생기지 않습니다.

**내 기록**

- 처음 사용한 입력과 기대값
- 실제 실행 결과와 달랐던 지점
- 수정한 줄 또는 설정, 다시 실행한 결과

## 이전 판 실습 085 · 자기 하네스로 기능 변경 끝내기

예상 120분. 자료를 준비하고 입력을 예측하는 데 15분, 첫 구현에 50분, 다른 입력과 실패 사례에 35분, 결과를 다시 확인하는 데 20분을 씁니다.

**준비물** 로컬 Codex CLI, Git, Jupyter, daily_resources/codex-kit. 80~85일은 같은 my-codex-desk를 사용한다.

[전체 실습 묶음](https://langchain-lecture.vercel.app/daily/all-labs.zip)을 풀고 이 노트북을 daily_resources 옆에 두세요. [오늘의 수업 예제](https://langchain-lecture.vercel.app/codex85/index.html)

### 내 하네스로 변경 요청을 처음부터 끝까지 처리하기

80~84일 저장소를 같은 폴더에서 연다. 업무 요청은 “동명 항목이 세 개일 때 3번만 완료한다”이다. 학생이 새 실패 테스트와 스킬의 완료 기준을 먼저 작성한 뒤 Codex에 요청한다. 입력 예시가 바뀌어도 역할·도구·훅이 실제로 작동하는지 확인한다.

In [ ]:
from pathlib import Path
import sys, json, shutil, subprocess
kit = Path('daily_resources/codex-kit').resolve()
repo = Path('my-codex-desk').resolve()
repo.mkdir(exist_ok=True)
def create_file(relative, content):
    target = repo / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_text(content, encoding='utf-8')
    return target
tasks_path = repo/'tasks.json'
if tasks_path.is_file():
    items = json.loads(tasks_path.read_text(encoding='utf-8'))
    if not any(row['id'] == 3 for row in items):
        items.append(dict(id=3,title=items[0]['title'],done=False))
        tasks_path.write_text(json.dumps(items,ensure_ascii=False,indent=2),encoding='utf-8')
    print('작업 원본 ID', [row['id'] for row in items])
expected = ['AGENTS.md','.agents/skills/todo-change/SKILL.md','.codex/hooks.json',
            '.codex/agents/case_reader.toml','.codex/config.toml','test_todo.py']
print('준비할 파일', [p for p in expected if not (repo/p).is_file()])
if (repo/'test_todo.py').exists():
    run = subprocess.run([sys.executable,'-m','unittest','-v'],cwd=repo,capture_output=True,text=True)
    print(run.stderr)

새 세션의 훅 기록, $todo-change 선택, case_reader 인계, get_task의 기존 ID 조회, 부모의 변경 diff와 회귀 테스트를 한 요청에 연결한다. 승인자 역할은 학생이다. 학생이 변경 계획을 읽고 적용을 승인한 뒤 진행한다. 거절할 요청에서는 먼저 작업 파일 세 개의 내용을 저장하고 거절 후 동일한지 검사한다. 세션 훅 이벤트는 별도로 증가할 수 있다. 이 대화상 승인과 64일의 프로그램 수준 쓰기 차단은 서로 다른 보장이다.

### 실행 경로

```mermaid
flowchart TD
  S["세션 시작 훅"] --> P["부모가 스킬 선택"]
  P --> D["서브에이전트 재현 보고"]
  P --> M["MCP로 업무 ID 조회"]
  D --> G{"학생이 계획 확인"}
  M --> G
  G -->|승인| T["부모 수정 후 테스트"]
  G -->|거절| N["수정 없이 종료"]
```

### 납품할 결과

삼중 동명 업무에서 고객이 지정한 ID만 완료하는 변경을 내 하네스로 납품합니다.

**구현 단서** 작업용 tasks.json에 세 번째 동명 항목(id=3, done=false)을 추가하고 세 항목을 모두 미완료로 초기화한 테스트를 준비하세요. 83일에 메모리 안의 완료 기능을 이미 고쳤습니다. 이번에는 complete_and_save(path, item_id)를 새로 만들어 파일을 다시 열어도 선택 결과가 남아야 합니다. 시험용 복사본에서 먼저 거절 경로, 이어 승인 경로를 실행합니다.

### 1. 구현하기

파일을 다시 열어도 세 번째 동명 항목만 완료 상태로 남도록 저장 기능의 새 테스트를 만든다. 부모에게 스킬·서브에이전트·MCP를 사용하도록 요청하고 변경 전 계획을 확인한다.

In [ ]:
validator = Path('daily_resources/cases/check_saved_todo.py').resolve()
print('터미널 검사 명령 인자', [sys.executable,str(validator),str(repo)])
# 먼저 실행해 미구현 함수 오류를 재현합니다. 기능 구현 뒤 같은 명령으로 저장 파일을 검산하세요.

### 2. 조건을 바꿔 다시 실행하기

학생이 계획을 승인한 경우와 거절한 경우를 다른 요청으로 실행하고 이벤트 로그·인계·도구 기록·diff를 비교한다.

In [ ]:
from hashlib import sha256
def work_files():
    return {name:sha256((repo/name).read_bytes()).hexdigest()
            for name in ('todo.py','tasks.json','test_todo.py')}
# 거절 요청을 보내기 직전에 before = work_files() 를 실행하세요.
# 거절한 뒤 assert work_files() == before 로 확인하세요. hook-events.jsonl은 별도로 읽습니다.

### 3. 스스로 확인하기

승인 시 선택한 ID만 바뀌고 회귀 검사가 통과한다. 거절 시 작업 파일이 바뀌지 않는다.

**실패 원인 좁히기** 거절 전후 todo.py·tasks.json·test_todo.py의 내용이 같아야 합니다. 세션 훅 로그는 늘어날 수 있습니다. 승인 후 완료 목록은 [3], 기존 2번 회귀도 통과해야 합니다. 저장소 옆에서 python daily_resources/cases/check_saved_todo.py my-codex-desk를 실행하면 시험용 복사본에서 두 ID를 각각 저장하고 다시 읽습니다.

**내 기록**

- 처음 사용한 입력과 기대값
- 실제 실행 결과와 달랐던 지점
- 수정한 줄 또는 설정, 다시 실행한 결과